In [1]:
# =============================================================================
# ADVANCED SUBMISSION: imgsz=800 + TTA + Weighted Boxes Fusion (WBF)
# =============================================================================
import os
import sys

# --- 1. Install Dependencies FIRST (Critical for Kaggle Save & Run All) ---
print("Installing dependencies...")
os.system("pip install -q ultralytics ensemble-boxes")

import numpy as np
import pandas as pd
import time
import cv2
from pathlib import Path
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# --- 2. Setup Paths ---
BEST_PT = Path("/kaggle/input/datasets/suchokkateryna/yolo-outputs/runs/detect/visdrone_yolo/yolov8l_800/weights/best.pt")
COMP_DIR = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
OUT_DIR = Path("/kaggle/working/artifacts_wbf")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_CSV = COMP_DIR / "test_images.csv"
test_df = pd.read_csv(TEST_CSV)

print("Loading model...")
model = YOLO(str(BEST_PT))

# --- 3. TTA Inference Function ---
def run_inference(img_path, flip=False):
    """Run inference on an image, optionally horizontally flipped."""
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    
    if flip:
        img = cv2.flip(img, 1)
    
    res = model.predict(
        source=img,
        imgsz=800,       # Match training resolution
        conf=0.20,       # Your winning threshold
        iou=0.45,        # Standard NMS before fusion
        max_det=1000,    # Allow more boxes before fusion
        device=0,
        verbose=False,
    )[0]

    if res.boxes is None or len(res.boxes) == 0:
        return np.zeros((0, 4)), np.zeros((0,)), np.zeros((0,))
    
    boxes = res.boxes.xyxy.cpu().numpy().astype(float)
    scores = res.boxes.conf.cpu().numpy()
    labels = res.boxes.cls.cpu().numpy()

    if flip:
        # Correctly flip boxes back to original coordinates
        new_x1 = w - boxes[:, 2]
        new_x2 = w - boxes[:, 0]
        boxes[:, 0] = new_x1
        boxes[:, 2] = new_x2
    
    return boxes, scores, labels

# --- 4. Run Pipeline ---
print(f"Generating submission for {len(test_df)} frames with TTA + WBF...")
sub_rows = []
t0 = time.time()

for i, (_, row) in enumerate(test_df.iterrows(), start=1):
    img_path = COMP_DIR / row["file_name"]
    if not img_path.exists():
        img_path = Path(row["file_name"])

    # Run Original and Flipped
    boxes_orig, scores_orig, labels_orig = run_inference(img_path, flip=False)
    boxes_flip, scores_flip, labels_flip = run_inference(img_path, flip=True)

    img_h, img_w = cv2.imread(str(img_path)).shape[:2]
    
    # Normalize boxes to [0, 1] for WBF
    boxes_list = []
    for b in [boxes_orig, boxes_flip]:
        if len(b) > 0:
            b_norm = b.copy()
            b_norm[:, 0] /= img_w
            b_norm[:, 1] /= img_h
            b_norm[:, 2] /= img_w
            b_norm[:, 3] /= img_h
            b_norm = np.clip(b_norm, 0, 1)
            boxes_list.append(b_norm)
        else:
            boxes_list.append(np.zeros((0, 4)))

    scores_list = [scores_orig, scores_flip]
    labels_list = [labels_orig, labels_flip]

    # Fuse with Weighted Boxes Fusion
    if len(boxes_orig) > 0 or len(boxes_flip) > 0:
        boxes_fused, scores_fused, labels_fused = weighted_boxes_fusion(
            boxes_list,
            scores_list,
            labels_list,
            iou_thr=0.55,
            skip_box_thr=0.001,
            conf_type='avg',
            allows_overflow=False
        )
        # Denormalize back to pixels
        boxes_fused[:, 0] *= img_w
        boxes_fused[:, 1] *= img_h
        boxes_fused[:, 2] *= img_w
        boxes_fused[:, 3] *= img_h
    else:
        boxes_fused, scores_fused, labels_fused = np.zeros((0,4)), np.zeros((0,)), np.zeros((0,))

    # Format to Kaggle string
    parts = []
    if len(scores_fused) > 0:
        order = np.argsort(-scores_fused)[:500]
        for idx in order:
            x1, y1, x2, y2 = boxes_fused[idx]
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0: continue
            parts.append(f"{int(labels_fused[idx])} {scores_fused[idx]:.4f} {x1:.1f} {y1:.1f} {w:.1f} {h:.1f}")
    
    sub_rows.append({"image_id": row["image_id"], "PredictionString": " ".join(parts)})

    if i % 500 == 0:
        elapsed = (time.time() - t0) / 60
        print(f"[{i}/{len(test_df)}] elapsed={elapsed:.1f}min")

# --- 5. Save ---
sub_df = pd.DataFrame(sub_rows)
sub_path = OUT_DIR / "submission_wbf_tta.csv"
sub_df.to_csv(sub_path, index=False)
print(f"\n✅ Saved: {sub_path}")
print(f"Non-empty: {(sub_df['PredictionString'].str.len() > 0).sum()}/{len(sub_df)}")

Installing dependencies...
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Loading model...
Generating submission for 3811 frames with TTA + WBF...
[500/3811] elapsed=1.9min
[1000/3811] elapsed=3.4min
[1500/3811] elapsed=4.9min
[2000/3811] elapsed=6.4min
[2500/3811] elapsed=8.0min
[3000/3811] elapsed=9.5min
[3500/3811] elapsed=11.1min

✅ Saved: /kaggle/working/artifacts_wbf/submission_wbf_tta.csv
Non-empty: 3811/3811
